# 03b Embeddings v2

This notebook builds the fixed embedding datasets, addressing the main problems with the original `03_embeddings.ipynb` pipeline:

| Problem | Fix |
|---|---|
| GraphSAGE link-prediction pretext has no connection to systemic risk | Feature-reconstruction loss: the GNN learns to encode financial node characteristics |
| GraphSAGE bottleneck too loose (64 ≈ 70 input dims, barely compresses) | 32-dim output — captures 97.7% of variance, forces meaningful compression |
| Node2Vec embedding dimension not tuned | 32-dim structural embeddings (down from 64) — same scale as GraphSAGE for fair comparison |
| Embedding spaces re-randomised each quarter | Warm-starting: each quarter initialises from the previous quarter's weights |

Both models remain **purely graph-structural** — no financial features are added — keeping the comparison against classical centrality measures fair.

Output files (same location and format as the originals):
- `src/data/embeddings/graphsage_fixed_srisk_dataset.parquet`
- `src/data/embeddings/node2vec_fixed_srisk_dataset.parquet`

In [1]:
import sys
import os
from pathlib import Path

%matplotlib inline

sys.path.insert(0, os.path.abspath('../..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.models.fix_embeddings import FixedGNNConfig, build_fixed_pooled_dataset
from src.models.embeddings import Node2VecConfig

PROJECT_ROOT = Path().resolve().parents[1]
DATA_PATH = PROJECT_ROOT / 'src' / 'datasets'
OUTPUT_ROOT = PROJECT_ROOT / 'src' / 'data'

pd.set_option('display.max_columns', 200)

## Configuration

In [2]:
# Fixed GraphSAGE: reconstruction loss instead of link prediction.
# 32-dim bottleneck: PCA shows 32 dims capture 97.7% of variance in the
# 70 financial node features — a meaningful compression that forces the
# GNN to learn a useful representation.
cfg_graphsage_fixed = FixedGNNConfig(
    hidden_dims=(256, 32),
    dropout=0.3,
    lr=0.01,
    epochs=100,
    reconstruction_weight=1.0,
    link_weight=0.0,
    aggregation="mean",
    device="cpu",
)

# Node2Vec v2: purely structural, same as v1 but with 32-dim embeddings
# instead of 64. Keeps the comparison against classical centrality fair
# (graph-only), and puts both embedding models on the same dimensional scale.
# Increase embedding_dim (e.g. 128) if you want a richer structural representation.
cfg_node2vec_fixed = Node2VecConfig(
    embedding_dim=32,
    walk_length=20,
    context_size=10,
    walks_per_node=10,
    num_negative_samples=1,
    batch_size=128,
    lr=0.01,
    epochs=100,
    device="cpu",
)

cfg_node2vec_fixed_v3 = Node2VecConfig(
    embedding_dim=512,
    walk_length=20,
    context_size=10,
    walks_per_node=10,
    num_negative_samples=1,
    batch_size=128,
    lr=0.01,
    epochs=100,
    device="cpu",
)

TARGET_COL = "log_systemic_risk_label"
INCLUDE_RAW_FEATURES = False

OUTPUT_GRAPHSAGE_FIXED = PROJECT_ROOT / "src" / "data" / "embeddings" / "graphsage_fixed_srisk_dataset.parquet"
OUTPUT_NODE2VEC_FIXED  = PROJECT_ROOT / "src" / "data" / "embeddings" / "node2vec_fixed_srisk_dataset.parquet"
OUTPUT_NODE2VEC_FIXED_V3  = PROJECT_ROOT / "src" / "data" / "embeddings" / "node2vec_fixed_v3_srisk_dataset.parquet"

OUTPUT_GRAPHSAGE_FIXED, OUTPUT_NODE2VEC_FIXED, OUTPUT_NODE2VEC_FIXED_V3

(WindowsPath('C:/Users/ruben/Desktop/Universidade/Nova IMS/Tese/Thesis/src/data/embeddings/graphsage_fixed_srisk_dataset.parquet'),
 WindowsPath('C:/Users/ruben/Desktop/Universidade/Nova IMS/Tese/Thesis/src/data/embeddings/node2vec_fixed_srisk_dataset.parquet'),
 WindowsPath('C:/Users/ruben/Desktop/Universidade/Nova IMS/Tese/Thesis/src/data/embeddings/node2vec_fixed_v3_srisk_dataset.parquet'))

# Build Fixed Embedding Datasets

## Fixed GraphSAGE

For each quarter, a `ReconstructionGNN` is trained on that quarter's graph.
The model consists of a GraphSAGE encoder and an MLP decoder. Training minimises
the MSE between the decoder's output and the original node features.

Each quarter is warm-started from the previous quarter's weights so that the
embedding space is temporally coherent across the full 2016–2023 panel.

In [3]:
pooled_df_graphsage_fixed = build_fixed_pooled_dataset(
    config=cfg_graphsage_fixed,
    years=range(2016, 2024),
    quarters=(1, 2, 3, 4),
    target_col=TARGET_COL,
    include_raw_features=INCLUDE_RAW_FEATURES,
    output_path=OUTPUT_GRAPHSAGE_FIXED,
)

pooled_df_graphsage_fixed.shape

Loaded 2016 Q1: 4548 banks, 11631 edges
Loaded 2016 Q2: 4548 banks, 11632 edges
Loaded 2016 Q3: 4548 banks, 11937 edges
Loaded 2016 Q4: 4548 banks, 11938 edges
Loaded 2017 Q1: 4548 banks, 11939 edges
Loaded 2017 Q2: 4548 banks, 11940 edges
Loaded 2017 Q3: 4548 banks, 11941 edges
Loaded 2017 Q4: 4548 banks, 11981 edges
Loaded 2018 Q1: 4548 banks, 12416 edges
Loaded 2018 Q2: 4548 banks, 12417 edges
Loaded 2018 Q3: 4548 banks, 12418 edges
Loaded 2018 Q4: 4548 banks, 12419 edges
Loaded 2019 Q1: 4548 banks, 12420 edges
Loaded 2019 Q2: 4548 banks, 12421 edges
Loaded 2019 Q3: 4548 banks, 12422 edges
Loaded 2019 Q4: 4548 banks, 12423 edges
Loaded 2020 Q1: 4548 banks, 12424 edges
Loaded 2020 Q2: 4548 banks, 12451 edges
Loaded 2020 Q3: 4548 banks, 12452 edges
Loaded 2020 Q4: 4548 banks, 12453 edges
Loaded 2021 Q1: 4548 banks, 12454 edges
Loaded 2021 Q2: 4548 banks, 12455 edges
Loaded 2021 Q3: 4548 banks, 12456 edges
Loaded 2021 Q4: 4548 banks, 12457 edges
Loaded 2022 Q1: 4548 banks, 12458 edges


(145536, 37)

In [4]:
pooled_df_graphsage_fixed.head()

,bank_id,year,quarter,period,emb_0,emb_1,emb_2,emb_3,emb_4,emb_5,emb_6,emb_7,emb_8,emb_9,emb_10,emb_11,emb_12,emb_13,emb_14,emb_15,emb_16,emb_17,emb_18,emb_19,emb_20,emb_21,emb_22,emb_23,emb_24,emb_25,emb_26,emb_27,emb_28,emb_29,emb_30,emb_31,log_systemic_risk_label
0,0,2016,1,2016Q1,17.404181,0.367222,4.288935,1.812043,4.624142,7.479827,18.412859,-2.293298,-9.765200,-14.006261,5.277770,-10.755105,7.813138,-13.013508,13.726084,23.227158,-6.386424,3.245816,4.768425,-15.940192,4.775862,16.480709,-7.079892,8.047514,16.867369,-15.176178,-0.588716,-11.425071,-15.028457,0.539563,-5.633980,5.936206,5.375278
1,1,2016,1,2016Q1,12.763929,-0.305598,1.996151,3.693714,2.065211,4.918036,17.245520,0.764605,-9.032034,-12.297436,4.464546,-7.429406,7.253392,-11.458472,10.945363,22.220997,-4.526939,2.728864,5.228599,-16.860556,4.113235,15.182696,-6.572977,5.463261,16.165442,-14.565371,6.048148,-10.303540,-13.987620,1.256087,-6.804884,5.583730,3.044522
2,2,2016,1,2016Q1,11.600013,-0.153854,2.578150,2.009159,3.198056,4.804305,13.654968,-0.903547,-7.452608,-10.385000,3.967799,-7.913086,5.529439,-9.068053,10.193670,17.140938,-4.243473,2.171646,3.728799,-11.760631,3.748623,12.420072,-4.726983,5.388383,12.349046,-11.745452,1.082221,-8.404320,-11.773826,0.247710,-4.167201,4.813051,4.564348
3,3,2016,1,2016Q1,12.982833,-0.048729,1.981491,3.830644,2.385361,5.600605,17.553280,0.784460,-9.488215,-12.981478,4.370619,-8.095097,7.543943,-11.755384,12.002787,22.408491,-4.720080,3.058141,5.797200,-17.241934,3.896642,15.124450,-6.302840,5.478939,16.805023,-15.513347,5.901018,-11.305252,-14.956190,0.923238,-6.765499,6.343685,3.637586
4,4,2016,1,2016Q1,3.490426,-0.282886,0.442768,1.803452,1.735125,1.766321,6.224274,0.629767,-3.939104,-5.319038,1.822969,-4.228253,2.339520,-3.480381,5.576026,7.473778,-1.405009,0.802032,2.613059,-5.349245,1.510372,5.542271,-1.311063,1.598758,5.950091,-6.902521,1.877847,-4.776368,-7.084520,-0.697089,-1.931310,3.574795,3.367296


In [5]:
embedding_cols = [c for c in pooled_df_graphsage_fixed.columns if c.startswith("emb_")]
meta_cols = ["bank_id", "year", "quarter", "period", TARGET_COL]
print(f"Total embedding columns: {len(embedding_cols)}  (32-dim bottleneck)")
meta_cols + embedding_cols[:5], len(embedding_cols)

Total embedding columns: 32  (32-dim bottleneck)


(['bank_id',
  'year',
  'quarter',
  'period',
  'log_systemic_risk_label',
  'emb_0',
  'emb_1',
  'emb_2',
  'emb_3',
  'emb_4'],
 32)

In [6]:
pooled_df_graphsage_fixed.groupby(["year", "quarter"])[TARGET_COL].agg(["count", "mean", "max"]).head(12)

count      mean       max
year quarter                           
2016 1         4548  0.719746  5.375278
     2         4548  0.714352  4.727388
     3         4548  0.711462  4.189655
     4         4548  0.709951  3.367296
2017 1         4548  0.710255  3.526361
     2         4548  0.708800  3.737670
     3         4548  0.708698  3.555348
     4         4548  0.709404  3.465736
2018 1         4548  0.713800  3.713572
     2         4548  0.711474  3.637586
     3         4548  0.709318  3.637586
     4         4548  0.709583  3.401197

## Node2Vec v2 (fixed)

Node2Vec v2 is purely structural — only the graph topology is used, no financial features are added. What changes vs v1:

- **32-dim embeddings** instead of 64, putting it on the same scale as the fixed GraphSAGE
- **Warm-starting** across quarters for temporal coherence

This keeps the comparison against classical centrality measures fair: both embedding models use only graph structure, just like PageRank, betweenness, and DebtRank do.

In [7]:
pooled_df_node2vec_fixed = build_fixed_pooled_dataset(
    config=cfg_node2vec_fixed,
    years=range(2016, 2024),
    quarters=(1, 2, 3, 4),
    target_col=TARGET_COL,
    include_raw_features=INCLUDE_RAW_FEATURES,
    output_path=OUTPUT_NODE2VEC_FIXED,
)

pooled_df_node2vec_fixed.shape

Loaded 2016 Q1: 4548 banks, 11631 edges
Loaded 2016 Q2: 4548 banks, 11632 edges
Loaded 2016 Q3: 4548 banks, 11937 edges
Loaded 2016 Q4: 4548 banks, 11938 edges
Loaded 2017 Q1: 4548 banks, 11939 edges
Loaded 2017 Q2: 4548 banks, 11940 edges
Loaded 2017 Q3: 4548 banks, 11941 edges
Loaded 2017 Q4: 4548 banks, 11981 edges
Loaded 2018 Q1: 4548 banks, 12416 edges
Loaded 2018 Q2: 4548 banks, 12417 edges
Loaded 2018 Q3: 4548 banks, 12418 edges
Loaded 2018 Q4: 4548 banks, 12419 edges
Loaded 2019 Q1: 4548 banks, 12420 edges
Loaded 2019 Q2: 4548 banks, 12421 edges
Loaded 2019 Q3: 4548 banks, 12422 edges
Loaded 2019 Q4: 4548 banks, 12423 edges
Loaded 2020 Q1: 4548 banks, 12424 edges
Loaded 2020 Q2: 4548 banks, 12451 edges
Loaded 2020 Q3: 4548 banks, 12452 edges
Loaded 2020 Q4: 4548 banks, 12453 edges
Loaded 2021 Q1: 4548 banks, 12454 edges
Loaded 2021 Q2: 4548 banks, 12455 edges
Loaded 2021 Q3: 4548 banks, 12456 edges
Loaded 2021 Q4: 4548 banks, 12457 edges
Loaded 2022 Q1: 4548 banks, 12458 edges


(145536, 37)

In [8]:
pooled_df_node2vec_fixed.head()

,bank_id,year,quarter,period,emb_0,emb_1,emb_2,emb_3,emb_4,emb_5,emb_6,emb_7,emb_8,emb_9,emb_10,emb_11,emb_12,emb_13,emb_14,emb_15,emb_16,emb_17,emb_18,emb_19,emb_20,emb_21,emb_22,emb_23,emb_24,emb_25,emb_26,emb_27,emb_28,emb_29,emb_30,emb_31,log_systemic_risk_label
0,0,2016,1,2016Q1,1.988349,7.224618,-0.088814,-0.569694,-1.907796,-1.858234,5.022929,0.209091,0.444999,-3.697957,0.787589,0.292406,0.697392,-6.654856,0.944105,-0.025433,0.315670,-4.740902,6.650463,2.548434,-6.265399,2.692690,-2.279794,-4.114759,0.835580,1.033708,-1.508416,0.391426,-3.656895,-2.878380,0.273383,0.942517,5.375278
1,1,2016,1,2016Q1,0.255643,2.902460,1.038193,0.496683,-2.322057,-1.083192,2.172666,-0.662479,0.575992,-5.060715,-0.168462,0.076775,-0.226057,-2.019874,0.516837,0.379725,-0.092547,-2.001682,4.308813,1.197535,-5.643735,1.072282,-1.629592,-2.153261,1.205504,-0.029262,-0.902266,0.993878,-2.152300,-1.718932,0.893029,-0.421778,3.044522
2,2,2016,1,2016Q1,0.827370,3.340770,0.862759,0.604033,-1.759995,-2.053487,1.885273,-0.198463,0.215771,-5.257910,0.132453,0.418528,-0.339741,-5.673204,-0.137251,0.132624,0.153582,-2.411833,6.141338,1.162140,-2.471608,1.335259,-3.260156,-5.441176,0.690045,0.700954,-1.087205,0.401503,-1.925390,0.767434,-0.259922,0.411519,4.564348
3,3,2016,1,2016Q1,0.018752,1.304138,2.360165,-0.985527,-1.290731,-3.459358,1.069472,-0.114857,-0.477913,-3.062650,0.945819,0.498186,-0.054182,-2.763167,-0.696853,0.531120,0.855149,-4.073358,4.374956,2.152529,-2.845438,-0.056007,-0.378247,-0.332774,2.225887,0.607371,-0.468703,1.236480,-2.354156,-1.354214,0.324954,0.644250,3.637586
4,4,2016,1,2016Q1,1.074310,0.008010,1.838244,0.273976,-1.002822,-0.830195,0.031323,-1.772162,-0.407490,-3.784968,0.461377,0.360469,-0.048202,-2.373918,0.111799,1.207784,-0.429389,-1.549426,0.394584,0.718478,-3.636705,1.558493,-2.182772,-3.098793,2.719436,0.925734,0.054504,0.577776,-0.805371,-2.594106,1.192160,1.409736,3.367296


In [9]:
embedding_cols_n2v = [c for c in pooled_df_node2vec_fixed.columns if c.startswith("emb_")]
meta_cols = ["bank_id", "year", "quarter", "period", TARGET_COL]
print(f"Total embedding columns: {len(embedding_cols_n2v)}  (32-dim structural, graph-only)")
meta_cols + embedding_cols_n2v[:5], len(embedding_cols_n2v)

Total embedding columns: 32  (32-dim structural, graph-only)


(['bank_id',
  'year',
  'quarter',
  'period',
  'log_systemic_risk_label',
  'emb_0',
  'emb_1',
  'emb_2',
  'emb_3',
  'emb_4'],
 32)

In [10]:
pooled_df_node2vec_fixed.groupby(["year", "quarter"])[TARGET_COL].agg(["count", "mean", "max"]).head(12)

count      mean       max
year quarter                           
2016 1         4548  0.719746  5.375278
     2         4548  0.714352  4.727388
     3         4548  0.711462  4.189655
     4         4548  0.709951  3.367296
2017 1         4548  0.710255  3.526361
     2         4548  0.708800  3.737670
     3         4548  0.708698  3.555348
     4         4548  0.709404  3.465736
2018 1         4548  0.713800  3.713572
     2         4548  0.711474  3.637586
     3         4548  0.709318  3.637586
     4         4548  0.709583  3.401197

In [11]:
cfg_node2vec_fixed_v3 = Node2VecConfig(
    embedding_dim=128,
    walk_length=20,
    context_size=10,
    walks_per_node=10,
    num_negative_samples=5,
    batch_size=128,
    lr=0.01,
    epochs=100,
    device="cpu",
)


pooled_df_node2vec_fixed_v3 = build_fixed_pooled_dataset(
    config=cfg_node2vec_fixed_v3,
    years=range(2016, 2024),
    quarters=(1, 2, 3, 4),
    target_col=TARGET_COL,
    include_raw_features=INCLUDE_RAW_FEATURES,
    output_path=OUTPUT_NODE2VEC_FIXED_V3,
)

pooled_df_node2vec_fixed_v3.shape

Loaded 2016 Q1: 4548 banks, 11631 edges
Loaded 2016 Q2: 4548 banks, 11632 edges
Loaded 2016 Q3: 4548 banks, 11937 edges
Loaded 2016 Q4: 4548 banks, 11938 edges
Loaded 2017 Q1: 4548 banks, 11939 edges
Loaded 2017 Q2: 4548 banks, 11940 edges
Loaded 2017 Q3: 4548 banks, 11941 edges
Loaded 2017 Q4: 4548 banks, 11981 edges
Loaded 2018 Q1: 4548 banks, 12416 edges
Loaded 2018 Q2: 4548 banks, 12417 edges
Loaded 2018 Q3: 4548 banks, 12418 edges
Loaded 2018 Q4: 4548 banks, 12419 edges
Loaded 2019 Q1: 4548 banks, 12420 edges
Loaded 2019 Q2: 4548 banks, 12421 edges
Loaded 2019 Q3: 4548 banks, 12422 edges
Loaded 2019 Q4: 4548 banks, 12423 edges
Loaded 2020 Q1: 4548 banks, 12424 edges
Loaded 2020 Q2: 4548 banks, 12451 edges
Loaded 2020 Q3: 4548 banks, 12452 edges
Loaded 2020 Q4: 4548 banks, 12453 edges
Loaded 2021 Q1: 4548 banks, 12454 edges
Loaded 2021 Q2: 4548 banks, 12455 edges
Loaded 2021 Q3: 4548 banks, 12456 edges
Loaded 2021 Q4: 4548 banks, 12457 edges
Loaded 2022 Q1: 4548 banks, 12458 edges


(145536, 133)

In [12]:
pooled_df_node2vec_fixed_v3.head()

,bank_id,year,quarter,period,emb_0,emb_1,emb_2,emb_3,emb_4,emb_5,emb_6,emb_7,emb_8,emb_9,emb_10,emb_11,emb_12,emb_13,emb_14,emb_15,emb_16,emb_17,emb_18,emb_19,emb_20,emb_21,emb_22,emb_23,emb_24,emb_25,emb_26,emb_27,emb_28,emb_29,emb_30,emb_31,emb_32,emb_33,emb_34,emb_35,emb_36,emb_37,emb_38,emb_39,emb_40,emb_41,emb_42,emb_43,emb_44,emb_45,emb_46,emb_47,emb_48,emb_49,emb_50,emb_51,emb_52,emb_53,emb_54,emb_55,emb_56,emb_57,emb_58,emb_59,emb_60,emb_61,emb_62,emb_63,emb_64,emb_65,emb_66,emb_67,emb_68,emb_69,emb_70,emb_71,emb_72,emb_73,emb_74,emb_75,emb_76,emb_77,emb_78,emb_79,emb_80,emb_81,emb_82,emb_83,emb_84,emb_85,emb_86,emb_87,emb_88,emb_89,emb_90,emb_91,emb_92,emb_93,emb_94,emb_95,emb_96,emb_97,emb_98,emb_99,emb_100,emb_101,emb_102,emb_103,emb_104,emb_105,emb_106,emb_107,emb_108,emb_109,emb_110,emb_111,emb_112,emb_113,emb_114,emb_115,emb_116,emb_117,emb_118,emb_119,emb_120,emb_121,emb_122,emb_123,emb_124,emb_125,emb_126,emb_127,log_systemic_risk_label
0,0,2016,1,2016Q1,-3.117326,-0.221066,-0.182117,-0.024927,0.788043,-1.722280,4.509021,0.337196,0.487897,-1.024630,1.917036,0.411994,-0.217111,-0.921758,0.128485,-0.118319,0.417805,1.877989,5.034539,-0.420347,1.959646,-1.216522,0.942113,-0.997906,0.003082,0.211298,0.667117,-0.208477,0.938085,-0.268511,-0.463426,-0.259241,-0.316172,-1.173883,0.831416,-1.119738,-1.974234,-1.917675,0.594680,0.037646,-0.154098,-5.988733,-3.763395,-0.047570,0.705523,-0.155384,-0.017563,0.940010,-0.094811,0.193685,-0.780507,0.204196,0.225641,-1.579758,7.683289,0.067030,-2.710745,-0.064522,0.525403,-0.026657,-2.054210,0.008808,0.643574,1.943400,1.776237,-0.224288,0.128074,-0.153309,5.758245,0.409852,0.604343,0.183722,0.216669,1.112893,0.313481,1.302702,1.755485,-1.331552,-0.406311,-0.240323,-0.391701,-0.176776,-0.203270,-0.095172,3.938536,0.338664,0.158336,0.956247,0.199973,0.091072,0.046257,-0.029355,0.405626,0.946375,0.093811,-1.489091,-0.380984,-0.161104,-0.923700,-0.252396,1.274286,-0.606002,-0.302405,0.706526,0.190331,0.078750,-0.304861,0.386964,0.026503,-0.310256,-0.060525,0.358725,0.075329,0.022942,0.034319,-0.164329,-0.335367,0.023864,1.134861,0.054183,-0.083483,-0.567694,-0.392361,1.001321,-0.093036,-0.155585,-0.256709,-0.479842,5.375278
1,1,2016,1,2016Q1,-1.027979,0.602625,0.442907,-0.326511,-0.601189,-1.602975,3.327982,0.518299,-0.720596,-2.201664,-0.223006,0.486147,-0.293256,-0.425904,1.206233,-0.192367,0.450396,0.549751,2.996444,-0.101708,1.002780,-0.326128,0.550620,-0.796647,-0.351261,0.130515,-0.095252,-0.612521,0.782713,-0.457235,0.066646,0.079800,-0.038315,-0.093491,-0.095610,-0.383137,0.077099,-1.168447,-0.140844,-1.588543,-0.503547,-1.471677,-2.339718,0.502238,0.637488,0.103755,-0.580530,-0.215435,0.120332,-0.024075,0.089526,0.051866,1.186656,-0.657946,2.993295,0.122722,-0.063688,0.257004,0.220463,-0.650464,-0.051290,-0.021118,-0.133679,4.758263,0.054990,-0.016433,0.602469,-0.688374,1.733496,0.460852,0.196060,0.223854,0.135900,1.410529,-0.067955,-0.760685,2.753568,-0.826322,0.572355,0.900555,0.144685,-0.301797,-0.184705,1.073496,0.720191,-0.589736,1.348139,-0.598043,1.189095,0.292037,1.782362,-0.040742,-0.049225,-1.189941,0.012054,-2.931920,-0.620276,-0.343172,-0.509037,0.579646,1.848426,-1.038010,-0.344771,0.635896,0.120891,0.354199,-1.173996,0.033170,0.673370,0.153009,-0.401781,-0.386525,0.060918,0.064100,0.140038,-0.580126,0.712643,0.087435,-0.207606,-0.229281,0.305081,0.201255,0.857649,-0.617638,-1.125984,-1.434801,-0.358732,-0.825848,3.044522
2,2,2016,1,2016Q1,-2.729951,0.711896,0.308759,0.494521,0.090314,-0.356784,1.690129,-0.376335,-0.483025,-1.599830,-0.285867,-0.608534,-0.339143,-0.485742,-0.219725,-0.093164,0.843319,1.887747,2.872154,-0.229604,2.551818,-1.104810,0.011693,-0.284518,0.113450,0.443220,0.706840,-0.422977,0.426736,0.769539,-0.312005,0.424683,0.201729,-0.061906,-0.039009,-1.049052,-1.700841,-0.717198,0.164815,0.317194,-1.059262,-2.669220,-2.535869,0.452195,0.071693,-0.057684,-0.116438,0.516595,0.023882,0.572364,-0.172868,-0.101113,0.347439,-0.675504,5.732213,-0.15

In [13]:
embedding_cols_n2v_v3 = [c for c in pooled_df_node2vec_fixed_v3.columns if c.startswith("emb_")]
meta_cols = ["bank_id", "year", "quarter", "period", TARGET_COL]
print(f"Total embedding columns: {len(embedding_cols_n2v_v3)}  (512-dim structural, graph-only)")
meta_cols + embedding_cols_n2v_v3[:5], len(embedding_cols_n2v_v3)

Total embedding columns: 128  (512-dim structural, graph-only)


(['bank_id',
  'year',
  'quarter',
  'period',
  'log_systemic_risk_label',
  'emb_0',
  'emb_1',
  'emb_2',
  'emb_3',
  'emb_4'],
 128)

In [14]:
pooled_df_node2vec_fixed_v3.groupby(["year", "quarter"])[TARGET_COL].agg(["count", "mean", "max"]).head(12)

count      mean       max
year quarter                           
2016 1         4548  0.719746  5.375278
     2         4548  0.714352  4.727388
     3         4548  0.711462  4.189655
     4         4548  0.709951  3.367296
2017 1         4548  0.710255  3.526361
     2         4548  0.708800  3.737670
     3         4548  0.708698  3.555348
     4         4548  0.709404  3.465736
2018 1         4548  0.713800  3.713572
     2         4548  0.711474  3.637586
     3         4548  0.709318  3.637586
     4         4548  0.709583  3.401197

# Exported Datasets

The saved datasets contain:
- metadata: `bank_id`, `year`, `quarter`, `period`
- target: `log_systemic_risk_label`
- embeddings: `emb_0`, `emb_1`, ..., `emb_31`

Files written by this notebook:
- `src/data/embeddings/graphsage_fixed_srisk_dataset.parquet` — 32-dim reconstruction-trained embeddings
- `src/data/embeddings/node2vec_fixed_srisk_dataset.parquet` — 32-dim purely structural embeddings

## What changed vs the originals

| | GraphSAGE original | **GraphSAGE fixed** | Node2Vec original | **Node2Vec fixed** |
|---|---|---|---|---|
| Objective | Link prediction | **Reconstruction** | Random walks | Random walks |
| Dim | 64 | **32** | 64 | **32** |
| Financial features | Used as GNN input | Used as GNN input | Ignored | Ignored |
| Temporal coherence | ✗ random init | **✓ warm-start** | ✗ random init | **✓ warm-start** |
| Graph-only | ✓ | ✓ | ✓ | ✓ |

## Next Step

Use the exported files in `05b_ML_Fixed_Embeddings.ipynb`.